In [ ]:
import time
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.graph import START, END, StateGraph
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
class GraphState(TypedDict):
    step1: str
    step2: str
    step3: str

In [ ]:
def step1(state: GraphState):
    print("step1 done")
    return {'step1': "done"}


def step2(state: GraphState):
    time.sleep(20)
    print("step2 done")
    return {'step2': "done"}


def step3(state: GraphState):
    print("step3 done")
    return {'step3': "done"}

In [ ]:
checkpointer = InMemorySaver()
graph = StateGraph(GraphState)

graph.add_node("step1", step1)
graph.add_node("step2", step2)
graph.add_node("step3", step3)

graph.add_edge(START, "step1")
graph.add_edge("step1", "step2")
graph.add_edge("step2", "step3")
graph.add_edge("step3", END)

workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
config = {"configurable": {"thread_id": "1"}}

workflow.invoke({}, config=config)

In [ ]:
list(workflow.get_state_history(config=config))

### Resume from where broke

In [ ]:
workflow.invoke(None, {"configurable": {"thread_id": '1', "checkpoint_id": "1f196eab-e87b-656e-8000-208b5f7cb7a0"}})

### Updating State

In [ ]:
workflow.update_state({"configurable": {"thread_id": '1', "checkpoint_id": "1f196e9a-d094-67f2-8003-2f31b426ec59"}}, {'step3': 'time travel'})